In [1]:
import sys
import numpy as np
from scipy.optimize import linprog
import itertools
import os
import time

In [2]:
import random
import math

In [3]:
random.seed(42)

In [4]:
tests = ['vrp_16_3_1', 'vrp_26_8_1', 'vrp_51_5_1', 'vrp_101_10_1', 'vrp_200_16_1', 'vrp_421_41_1']
thresholds = [(387, 280), (1019, 630), (713, 540), (1193, 830), (3719, 1400), (2392, 2000)]

In [7]:
def load_data(test):
    with open(f"data/{test}") as file:
        lines = file.readlines()
        args = lines[0].split()
        n = int(args[0])
        v = int(args[1])
        c = float(args[2])
        req = list()
        points = list()
        for i in range(n):
            d, x, y = list(map(float, lines[1 + i].split()))
            req.append(d)
            points.append((x, y))
            
        return n, v, c, req, points

In [58]:
def check_vrp(n, v, c, req, points, paths):
    used = [0] * n
    taken = [0] * v

    if len(paths) != v:
        return 1e18
    
    for wh, path in enumerate(paths):
        for i in path:
            taken[wh] += req[i - 1]
            used[i - 1] += 1

    for i in range(v):
        if taken[i] > c:
            return 1e18
            
    for i in range(n):
        if used[i] != 1:
            return 1e18
    res = 0
    for path in paths:
        if len(path) == 0:
            continue
        res += math.dist((0, 0), points[path[0] - 1])
        
        for i in range(len(path) - 1):
            res += math.dist(points[path[i] - 1], points[path[i + 1] - 1])
            
        res += math.dist(points[path[-1] - 1], (0, 0))

    return res

In [59]:
def passed_cnt(result, idx):
    if result <= thresholds[idx][1]:
        return 2
    elif result <= thresholds[idx][0]:
        return 1
    else:
        return 0

In [60]:
def test_method(method, name, use_file=False):
    print(f"Checking {name}")
    score = 0
    for i, test in enumerate(tests):
        n, v, c, req, points = load_data(test)
        start = time.time()
        
        if not use_file:
            paths = method(n, v, c, req, points)
        else:
            paths = method(test)

        end = time.time()
        elapsed = end - start
        print(f"Execution time: {elapsed:.4f} seconds")
            
        result = check_vrp(n, v, c, req, points, paths)
        passed = passed_cnt(result, i)
        
        if passed == 1:
            score += 3
        elif passed == 2:
            score += 5

        print(f"Target function {test}: {result}")
        print(f"Passed {test}: {passed}")

    print(f"Score: {score}")

Пока будем решать задачу в декомпозированном виде:
1. Найдем сопоставление каждого покупателя -- доставщику
2. Решим на доставщике TSP

В качестве жадного решения будем делать следующее: 
1. Поддерживаем циклы изначально каждая вершина цикл с 0.
2. Ищем два цикла, которые можно склеить и если это возможно, то склеиваем -- находим наилучшее склеивание.

In [64]:
!g++ -O2 -std=c++2a cpp_methods/greedy.cpp -o tmp/greedy

In [65]:
def greedy_vrp(test_file): 
    os.system(f"./tmp/greedy data/{test_file}")
    with open("tmp/ans.txt") as file:
        lines = file.readlines()
        decomp = list()
        for i in range(len(lines)):
            decomp.append(list(map(int, lines[i].split())))
        return decomp

In [66]:
test_method(greedy_vrp, "greedy_vrp", True)

Checking greedy_vrp
Execution time: 0.4011 seconds
Target function vrp_16_3_1: 467.68990576011885
Passed vrp_16_3_1: 0
Execution time: 0.0145 seconds
Target function vrp_26_8_1: 1e+18
Passed vrp_26_8_1: 0
Execution time: 0.0116 seconds
Target function vrp_51_5_1: 834.2331559060076
Passed vrp_51_5_1: 0
Execution time: 0.0101 seconds
Target function vrp_101_10_1: 1593.9601262891183
Passed vrp_101_10_1: 0
Execution time: 0.0162 seconds
Target function vrp_200_16_1: 1e+18
Passed vrp_200_16_1: 0
Execution time: 0.0591 seconds
Target function vrp_421_41_1: 1958.7462601014647
Passed vrp_421_41_1: 2
Score: 5


Интересно...

Как мы видим жадное решение получило отличный скор на последнем тесте, но на некоторых тестах выдает невалидные решения (не может склеивать циклы достаточное количество раз из - за превышения $\text{capacity}$ курьера).
Попробуем добавить локальную оптимизацию, где мы берем вершину из какого - то цикла и пытаемся ее добавить в другой цикл. Чтобы бороться с невалидными циклами добавим в стоимость решения превышение $\text{capacity}$, умноженное на $\lambda$.

In [70]:
!g++ -O2 -std=c++2a cpp_methods/local_opt.cpp -o tmp/local_opt

In [71]:
def local_opt_vrp(test_file): 
    os.system(f"./tmp/local_opt data/{test_file}")
    with open("tmp/ans.txt") as file:
        lines = file.readlines()
        decomp = list()
        for i in range(len(lines)):
            decomp.append(list(map(int, lines[i].split())))
        return decomp

In [72]:
test_method(local_opt_vrp, "local_opt_vrp", True)

Checking local_opt_vrp
Execution time: 0.4771 seconds
Target function vrp_16_3_1: 467.68990576011885
Passed vrp_16_3_1: 0
Execution time: 0.0232 seconds
Target function vrp_26_8_1: 1e+18
Passed vrp_26_8_1: 0
Execution time: 0.0133 seconds
Target function vrp_51_5_1: 828.6085597862906
Passed vrp_51_5_1: 0
Execution time: 0.0119 seconds
Target function vrp_101_10_1: 1592.3048438228159
Passed vrp_101_10_1: 0
Execution time: 0.0295 seconds
Target function vrp_200_16_1: 2424.373787543716
Passed vrp_200_16_1: 1
Execution time: 0.1634 seconds
Target function vrp_421_41_1: 1953.8798188912651
Passed vrp_421_41_1: 2
Score: 8


Прошел еще один тест